In [0]:
dbutils.widgets.text("landing_time","")
dbutils.widgets.text("table_name","")
dbutils.widgets.text("merge_keys","")
dbutils.widgets.text("load_types","")

landing_time = dbutils.widgets.get("landing_time")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_types = dbutils.widgets.get("load_types")

In [0]:
bronaze_df = spark.read.table(f"adb_databricks_careflow.bronze.{table_name}").filter(f"landing_time = '{landing_time}'")

In [0]:
## _insert_time in bronze will be replace here by new col named as same i.e "_insert_time"
## _update_time will be used for future purpose for MERGE kind of incremental load, so when data will be updated then update time will also be updated accordingly not the _insert_time

from pyspark.sql import functions as F

updated_df = bronaze_df.withColumn("_insert_time", F.current_timestamp()).withColumn("_update_time", F.current_timestamp()) 

In [0]:

from pyspark.sql import functions as F

merge_keys = [key.strip() for key in merge_keys.split(",") if key.strip()]
normal_load_type = (load_types or "FULL").strip().upper()
target_table = f"adb_databricks_careflow.silver.{table_name}"

if normal_load_type == "MERGE":
    if not merge_keys:
        raise ValueError("please provide the merge key")

    missing_merge_keys = []
    for key in merge_keys:
        if key not in bronaze_df.columns:
            missing_merge_keys.append(key)
            raise ValueError(f"please provide the correct merge key, {key} is not in bronze table")

updated_df = (
    bronaze_df.withColumn("_insert_time", F.current_timestamp()).withColumn("_update_time", F.current_timestamp())
)

In [0]:
if normal_load_type == "FULL":
    updated_df.write.mode("overwrite").saveAsTable(target_table)
elif normal_load_type == "APPEND":
    updated_df.write.mode("append").saveAsTable(target_table)
elif normal_load_type == "MERGE":
    target_exists = spark.catalog.tableExists(target_table)
    if not target_exists:
        updated_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        source_view = "silver_merge_source"
        updated_df.createOrReplaceTempView(source_view)

        merge_condition = " AND ".join(
            [f"target.`{key}` = source.`{key}`" for key in merge_keys]
        )
        update_assignments = ", ".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in updated_df.columns
                if column != "insert_timestamp"
            ]
        )
        insert_columns = ", ".join([f"`{column}`" for column in updated_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_df.columns])

        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING {source_view} AS source
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET {update_assignments}
            WHEN NOT MATCHED THEN INSERT ({insert_columns})
            VALUES ({insert_values})
        """.strip()

        print("[MERGE SQL]")
        print(merge_sql)

        spark.sql(merge_sql)
        print("[STEP 3 COMPLETE] MERGE finished successfully")
else:
    raise ValueError("load_types must be one of: FULL, APPEND, MERGE")
     


In [0]:
## Used `notebook.exit` to get the records count
## `dbutils.notebook.exit` stops notebook execution immediately and returns a single string value to a parent notebook or job

records_count = updated_df.count()
dbutils.notebook.exit(str(records_count))